# RoadScan — Run 16: RDD2022 checkpoint fine-tuning

Fine-tune SreekarAditya's published YOLOv8s RDD2022 seed-0 checkpoint on exactly the same **8,999 training images** and **1,977 China MotorBike validation images** as run 14. Reuse run 14's requested training settings, including 640 pixels, batch 16, seed 42, AdamW, augmentations, 80 maximum epochs, patience 15 and a three-hour epoch-boundary training budget. No personal-photo dataset is attached or opened.

Run 14 used YOLO11s initialized from COCO; run 16 uses YOLOv8s initialized from RDD2022. This compares two complete pipelines, not an isolated test of pretraining or architecture. External pretraining overlap with RDD2022 validation/US images is unknown; RDD2022 results are diagnostic development scores, not clean generalization evidence. No performance improvement is guaranteed.

Sources: [team run 14](https://www.kaggle.com/code/rimasabunasser/14-roadscan-team-v2-training?scriptVersionId=361531048), [published checkpoint](https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark), [Ultralytics training](https://docs.ultralytics.com/modes/train/).

## 1. Verify and stage the frozen run-14 data

Use the **saved output of run 14, version 1** as the only data input. Validate image-label pairs, box geometry, image decoding, original sample membership, country counts and exact cross-split duplicates. Match every image hash to run 14's frozen manifest. Images and labels are copied unchanged; no resampling or new split is performed. Keep US and personal photos outside this run.


In [1]:
from pathlib import Path
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, math, shutil, platform
import numpy as np
import pandas as pd
from PIL import Image
import yaml
from IPython.display import display, Markdown

BASE = Path('/kaggle/input/notebooks/rimasabunasser/14-roadscan-team-v2-training')
REF = BASE/'artifacts_team_v2'
assert (REF/'requested_training_settings.json').is_file(), 'Attach run 14 version 1 outputs.'
SOURCE = BASE/'team_v2_data'
ROOT = Path('/kaggle/working/run16_team_v2_data')
ART = Path('/kaggle/working/artifacts_run16')
ART.mkdir(exist_ok=True)
CLASSES = ['D00','D10','D20','D40']
assert SOURCE.is_dir(), 'Attach the original team sample9000 v2 notebook output.'
original_manifest = (REF/'original_team_sample9000_list.txt').read_text().splitlines()
assert len(original_manifest)==len(set(original_manifest))==8999
jobs=[]
for split,expected in [('train',8999),('val',1977)]:
    images={p.stem:p for p in (SOURCE/split/'images').glob('*.jpg')}
    labels={p.stem:p for p in (SOURCE/split/'labels').glob('*.txt')}
    assert images.keys()==labels.keys() and len(images)==expected, (split,len(images),len(labels))
    if split=='train':
        assert set(images)==set(original_manifest)
        assert Counter(s.split('_')[0] for s in images)=={'Japan':4000,'India':3500,'Czech':1499}
    else:
        assert all(s.startswith('China_MotorBike_') for s in images)
    for folder in ['images','labels']:
        (ROOT/split/folder).mkdir(parents=True,exist_ok=True)
    jobs.extend((split,stem,images[stem],labels[stem]) for stem in sorted(images))

def stage_and_audit(job):
    split,stem,image,label=job
    raw=label.read_text()
    present=[0]*4
    boxes=0
    for line in raw.splitlines():
        if not line.strip():
            continue
        row=list(map(float,line.split()))
        assert len(row)==5 and all(math.isfinite(x) for x in row),str(label)
        c,x,y,w,h=row
        assert c==int(c) and 0<=c<4 and 0<w<=1 and 0<h<=1,str(label)
        assert x-w/2>=-1e-4 and y-h/2>=-1e-4 and x+w/2<=1.0001 and y+h/2<=1.0001,str(label)
        present[int(c)]=1
        boxes+=1
    with Image.open(image) as im:
        im.load()
        width,height=im.size
    digest=hashlib.sha256(image.read_bytes()).hexdigest()
    for src,folder in [(image,'images'),(label,'labels')]:
        shutil.copy2(src,ROOT/split/folder/src.name)
    return dict(split=split,stem=stem,country=stem.split('_')[0],sha256=digest,width=width,height=height,boxes=boxes,empty=int(boxes==0),**dict(zip(CLASSES,present)))

records=[]
with ThreadPoolExecutor(max_workers=8) as pool:
    for i,row in enumerate(pool.map(stage_and_audit,jobs),1):
        records.append(row)
        if i%1000==0:
            print('Audited and staged:',i,'/',len(jobs),flush=True)
manifest=pd.DataFrame(records)
train_hash=set(manifest.loc[manifest.split.eq('train'),'sha256'])
val_hash=set(manifest.loc[manifest.split.eq('val'),'sha256'])
assert not train_hash & val_hash, 'Exact image leakage across train and validation.'
assert set(manifest.loc[manifest.split.eq('train'),'country']).isdisjoint(set(manifest.loc[manifest.split.eq('val'),'country']))
manifest.to_csv(ART/'manifest.csv',index=False)
shutil.copy2(REF/'original_team_sample9000_list.txt',ART/'original_team_sample9000_list.txt')
manifest_digest=hashlib.sha256((chr(10).join(sorted(original_manifest))+chr(10)).encode()).hexdigest()
(ART/'sample_manifest_sha256.txt').write_text(manifest_digest)
DATA=ROOT/'data.yaml'
DATA.write_text(yaml.safe_dump(dict(path=str(ROOT),train='train/images',val='val/images',names=dict(enumerate(CLASSES))),sort_keys=False))
shutil.copy2(DATA,ART/'data.yaml')
summary=manifest.groupby(['split','country']).agg(images=('stem','size'),empty=('empty','sum'),**{c:(c,'sum') for c in CLASSES}).reset_index()
summary.to_csv(ART/'data_summary.csv',index=False)
display(summary)
(ART/'protocol.json').write_text(json.dumps(dict(train=8999,validation=1977,train_countries=['Japan','India','Czech'],validation_country='China_MotorBike',sample_source=str(BASE),sample_sha256=manifest_digest,US_status='not opened in this run; external checkpoint pretraining exposure is unknown',local_images='test only; never train or tune',test_evaluated=False,selection='exact original team v2 sample; enriched for D10/D40',split='country-disjoint; exact hashes checked; no verified drive IDs',baseline_comparison='same image assignments; baseline implementation and threshold protocol must be verified before claiming a controlled comparison'),indent=2))
print('DATA_READY',flush=True)
display(Markdown('**Data interpretation.** The exact team v2 sample passed image decoding, label geometry, country separation and exact cross-split duplicate checks. Japan/India/Czech are training countries; China MotorBike is development validation. This preserves the team sample and does not use US or local test images. Sampling enriches rare classes, so training frequencies are not population prevalence.'))

frozen14 = pd.read_csv(REF/'manifest.csv')
assert manifest.set_index(['split','stem']).sha256.to_dict() == frozen14.set_index(['split','stem']).sha256.to_dict(), 'Image hashes differ from run 14.'
summary14 = json.loads((REF/'summary.json').read_text())
assert manifest_digest == summary14['sample_sha256']
label_hashes = [dict(split=split, stem=stem, sha256=hashlib.sha256(label.read_bytes()).hexdigest()) for split,stem,image,label in jobs]
pd.DataFrame(label_hashes).to_csv(ART/'label_sha256.csv',index=False)
display(Markdown('**Run-14 match.** All 10,976 image hashes and the 8,999-image training list match run 14. Labels are copied unchanged and their hashes are recorded. This verifies the team split, but cannot exclude overlap with the external model pretraining data.'))


Audited and staged: 1000 / 10976
Audited and staged: 2000 / 10976
Audited and staged: 3000 / 10976
Audited and staged: 4000 / 10976
Audited and staged: 5000 / 10976
Audited and staged: 6000 / 10976
Audited and staged: 7000 / 10976
Audited and staged: 8000 / 10976
Audited and staged: 9000 / 10976
Audited and staged: 10000 / 10976


,split,country,images,empty,D00,D10,D20,D40
0,train,Czech,1499,427,764,300,129,154
1,train,India,3500,500,1012,60,1586,1530
2,train,Japan,4000,500,989,2240,1741,1390
3,val,China,1977,43,1419,738,486,164


DATA_READY


**Data interpretation.** The exact team v2 sample passed image decoding, label geometry, country separation and exact cross-split duplicate checks. Japan/India/Czech are training countries; China MotorBike is development validation. This preserves the team sample and does not use US or local test images. Sampling enriches rare classes, so training frequencies are not population prevalence.

**Run-14 match.** All 10,976 image hashes and the 8,999-image training list match run 14. Labels are copied unchanged and their hashes are recorded. This verifies the team split, but cannot exclude overlap with the external model pretraining data.

## 2. Verify the published weights and match run-14 settings

Download the exact checkpoint used in run 15 and verify its SHA-256 and class order. Read the requested training settings directly from run 14 and allow only output/data path and run-name changes. Keep a fresh optimizer and schedule; this is fine-tuning, not resuming the publisher training session. Record provenance and the overlap limitation.


In [2]:
import sys, subprocess, importlib.metadata, time, urllib.request
EXPECTED_SHA256 = '565aae6915e34e4f7f4c44dac0bdcf9fff21b83c4adc74cc97ca09cc31882ce8'
WEIGHT_URL = 'https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark/resolve/main/yolo-rdd2022-benchmark/yolov8s_seed0_best.pt?download=true'
WEIGHTS = ART/'published_yolov8s_rdd2022_seed0.pt'
if not WEIGHTS.exists():
    request = urllib.request.Request(WEIGHT_URL, headers={'User-Agent':'RoadScan-research'})
    temporary = WEIGHTS.with_suffix('.download')
    with urllib.request.urlopen(request,timeout=180) as response, temporary.open('wb') as output:
        shutil.copyfileobj(response,output)
    assert hashlib.sha256(temporary.read_bytes()).hexdigest()==EXPECTED_SHA256, 'Checkpoint digest mismatch.'
    temporary.replace(WEIGHTS)
assert hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()==EXPECTED_SHA256
try:
    installed=importlib.metadata.version('ultralytics')
except importlib.metadata.PackageNotFoundError:
    installed=None
if installed!='8.4.177':
    subprocess.check_call([sys.executable,'-m','pip','install','-q','ultralytics==8.4.177'])
import torch, ultralytics
from ultralytics import YOLO
assert ultralytics.__version__=='8.4.177', 'Restart the session after changing Ultralytics.'
assert torch.cuda.is_available(), 'Select GPU T4 x2 before Save & Run All.'
model=YOLO(str(WEIGHTS))
published_names=[str(model.names[i]) for i in range(len(model.names))]
assert published_names in [CLASSES,['longitudinal_crack','transverse_crack','alligator_crack','pothole']], published_names
mapping=[dict(id=i,published_name=published_names[i],rdd_code=CLASSES[i]) for i in range(4)]
(ART/'class_mapping.json').write_text(json.dumps(mapping,indent=2))
(ART/'environment.json').write_text(json.dumps(dict(python=platform.python_version(),torch=torch.__version__,ultralytics=ultralytics.__version__,gpu=torch.cuda.get_device_name(0)),indent=2))
provenance=dict(run=16,model='YOLOv8s',initialization='published RDD2022 seed0',publisher='SreekarAditya',model_card='https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark',download=WEIGHT_URL,sha256=EXPECTED_SHA256,license='CC BY 4.0 as stated by publisher; retain SreekarAditya and RDD2022 attribution',external_training_overlap='unknown; not excluded',clean_generalization_claim_allowed=False,comparison_to_14='pipeline comparison; architecture and pretraining both differ',personal_photos_used=False,US_evaluated_in_this_run=False)
(ART/'provenance.json').write_text(json.dumps(provenance,indent=2))
settings14=json.loads((REF/'requested_training_settings.json').read_text())
settings=dict(settings14)
settings.update(data=str(DATA),project='/kaggle/working/runs_run16',name='yolov8s_rdd2022_640_seed42',exist_ok=False)
allowed_differences={'data','project','name'}
differences={k:dict(run14=settings14.get(k),run16=settings.get(k)) for k in set(settings14)|set(settings) if settings14.get(k)!=settings.get(k)}
assert set(differences)<=allowed_differences, differences
assert (settings['epochs'],settings['patience'],settings['batch'],settings['imgsz'],settings['seed'])==(80,15,16,640,42)
(ART/'requested_training_settings.json').write_text(json.dumps(settings,indent=2))
(ART/'settings_comparison_run14.json').write_text(json.dumps(dict(matching_hyperparameters=True,allowed_path_and_run_name_differences=differences,run14_model='YOLO11s / COCO',run16_model='YOLOv8s / RDD2022',training_budget_seconds=10800),indent=2))
display(pd.DataFrame(mapping))
display(Markdown('**Preparation interpretation.** The checkpoint matches the recorded SHA-256 and its four classes align with D00/D10/D20/D40. All explicit training hyperparameters match run 14; only data/output paths and the run name differ. Model architecture and initialization are intentionally different.'))
print('RUN16_READY',flush=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


,id,published_name,rdd_code
0,0,longitudinal_crack,D00
1,1,transverse_crack,D10
2,2,alligator_crack,D20
3,3,pothole,D40


**Preparation interpretation.** The checkpoint matches the recorded SHA-256 and its four classes align with D00/D10/D20/D40. All explicit training hyperparameters match run 14; only data/output paths and the run name differ. Model architecture and initialization are intentionally different.

RUN16_READY


## 3. Fine-tune YOLOv8s on the frozen team sample

Train with the verified run-14 settings: up to 80 epochs, patience 15 and a three-hour budget checked at epoch boundaries. Use GPU 0, matching run 14. Save best/last checkpoints and training history during and after training. Preparation and evaluation add time. The best checkpoint is selected on China validation.


In [3]:
clock={}
def start_clock(trainer):
    clock['start']=time.monotonic()
def budget_and_checkpoint(trainer):
    for src,name in [(Path(trainer.best),'best.pt'),(Path(trainer.last),'last.pt'),(Path(trainer.save_dir)/'results.csv','results.csv')]:
        if src.exists():
            shutil.copy2(src,ART/name)
    if time.monotonic()-clock['start']>=180*60:
        trainer.stop=True
        print('Three-hour training budget reached; stopping after this epoch.',flush=True)
model.add_callback('on_train_start',start_clock)
model.add_callback('on_fit_epoch_end',budget_and_checkpoint)
model.train(**settings)
RUN=Path(model.trainer.save_dir)
for src in [RUN/'weights/best.pt',RUN/'weights/last.pt',RUN/'results.csv',RUN/'args.yaml',RUN/'results.png']:
    if src.exists():
        shutil.copy2(src,ART/src.name)
history=pd.read_csv(RUN/'results.csv')
display(history.tail(5))
print('TRAINING_COMPLETE. Epochs:',len(history),flush=True)
display(Markdown('**Run 16 training interpretation.** The best checkpoint was selected using China validation detection fitness. The training schedule allows up to 80 epochs with patience 15 and a three-hour epoch-boundary budget. This run matches run 14 settings while changing the complete model initialization pipeline to RDD2022-pretrained YOLOv8s. More training does not guarantee higher accuracy; the evaluation below determines the result.'))


New https://pypi.org/project/ultralytics/8.4.190 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/run16_team_v2_data/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kag

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/80       4.3G      1.758      1.652      1.558         30        640: 100% ━━━━━━━━━━━━ 563/563 5.5it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 4.6it/s 13.6s
                   all       1977       4650      0.749        0.7      0.793      0.446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/80       4.3G      1.772      1.686       1.58         23        640: 100% ━━━━━━━━━━━━ 563/563 5.6it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 62/62 4.6it/s 13.5s
                   all       1977       4650      0.659      0.633      0.675      0.351

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/80       4.3G      1.772      1.695      1.572         20        640: 100% ━━━━━━━━━━━━ 563/563 5.6it/s 1:41
                 Class     Images  Instances      Box(

,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
12,13,1500.29,1.70459,1.52568,1.51584,0.56707,0.50622,0.50896,0.23879,1.90568,1.97318,1.71498,0.000946,0.000946,0.000946
13,14,1615.86,1.69954,1.51875,1.51748,0.63242,0.49619,0.54100,0.24497,2.00608,1.79958,1.80513,0.000937,0.000937,0.000937
14,15,1731.13,1.68891,1.50747,1.50879,0.52504,0.45158,0.44622,0.20475,2.08670,2.34801,1.87924,0.000927,0.000927,0.000927
15,16,1846.25,1.67836,1.48010,1.49298,0.53389,0.49476,0.46542,0.20277,2.06413,2.06623,1.84813,0.000917,0.000917,0.000917
16,17,1961.35,1.68064,1.47988,1.49977,0.58059,0.45549,0.46362,0.21501,1.98682,1.91043,1.78628,0.000905,0.000905,0.000905


TRAINING_COMPLETE. Epochs: 17


**Run 16 training interpretation.** The best checkpoint was selected using China validation detection fitness. The training schedule allows up to 80 epochs with patience 15 and a three-hour epoch-boundary budget. This run matches run 14 settings while changing the complete model initialization pipeline to RDD2022-pretrained YOLOv8s. More training does not guarantee higher accuracy; the evaluation below determines the result.

## 4. Evaluate the trained model

Use the same validation procedure as run 14: detection mAP, per-class AP, confidence curves, confusion matrices, fixed-0.5 image Macro-F1 and optimistic validation-tuned thresholds. Measure the same 20-image CPU latency sample. Save actual results to JSON and generate an English interpretation. No US or personal-photo evaluation is run.


In [4]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
best=YOLO(str(ART/'best.pt'))
det=best.val(data=str(DATA),split='val',imgsz=640,batch=16,device=0,workers=2,conf=0.001,iou=0.7,max_det=300,plots=True,project='/kaggle/working/evaluation_run16',name='china_development')
detection_metrics={str(k):float(v) for k,v in det.results_dict.items()}
(ART/'china_detection.json').write_text(json.dumps(detection_metrics,indent=2))
per_class=pd.DataFrame(dict(cls=CLASSES,AP50=[float(x) for x in det.box.ap50],AP50_95=[float(x) for x in det.box.ap]))
per_class.to_csv(ART/'china_per_class_detection.csv',index=False)
display(per_class)
part=manifest.loc[manifest.split.eq('val')].sort_values('stem')
stems=part.stem.tolist()
lookup={s:i for i,s in enumerate(stems)}
truth=part[CLASSES].to_numpy(dtype=int)
scores=np.zeros((len(stems),4),dtype=float)
for result in best.predict(source=str(ROOT/'val/images'),imgsz=640,conf=0.001,iou=0.7,max_det=300,device=0,stream=True,verbose=False):
    i=lookup[Path(result.path).stem]
    for c,p in zip(result.boxes.cls.cpu().numpy().astype(int),result.boxes.conf.cpu().numpy()):
        scores[i,c]=max(scores[i,c],float(p))
grid=np.arange(0.05,0.96,0.05)
thresholds=[]
for c in range(4):
    fs=[precision_recall_fscore_support(truth[:,c],scores[:,c]>=t,average='binary',zero_division=0)[2] for t in grid]
    thresholds.append(float(grid[int(np.argmax(fs))]))
rows=[]
macro={}
for mode,limits in [('fixed_0.5',np.full(4,0.5)),('validation_tuned_optimistic',np.array(thresholds))]:
    p,r,f,support=precision_recall_fscore_support(truth,scores>=limits,zero_division=0)
    macro[mode]=float(f.mean())
    for c,name in enumerate(CLASSES):
        prevalence=float(truth[:,c].mean())
        rows.append(dict(cls=name,mode=mode,threshold=float(limits[c]),precision=float(p[c]),recall=float(r[c]),f1=float(f[c]),support=int(support[c]),always_positive_f1=2*prevalence/(1+prevalence),auroc=float(roc_auc_score(truth[:,c],scores[:,c])) if len(np.unique(truth[:,c]))==2 else None))
image_metrics=pd.DataFrame(rows)
image_metrics.to_csv(ART/'china_image_metrics.csv',index=False)
np.savez_compressed(ART/'china_image_scores.npz',stems=np.array(stems),truth=truth,scores=scores)
display(image_metrics)
config=dict(classes=CLASSES,imgsz=640,conf_floor=0.001,iou=0.7,max_det=300,image_thresholds=thresholds,image_threshold_source='China validation; optimistic development selection',box_confidence=0.25,box_confidence_status='provisional; review validation precision/recall and F1-confidence curves before deployment')
(ART/'inference_config.json').write_text(json.dumps(config,indent=2))
# CPU latency includes model prediction and preprocessing, not app upload/retrieval.
cpu=YOLO(str(ART/'best.pt'))
torch.set_num_threads(2)
latency_paths=[ROOT/'val/images'/(s+'.jpg') for s in stems[:20]]
for _ in range(3):
    cpu.predict(str(latency_paths[0]),imgsz=640,conf=0.25,device='cpu',verbose=False)
latencies=[]
for image in latency_paths:
    start=time.perf_counter()
    cpu.predict(str(image),imgsz=640,conf=0.25,device='cpu',verbose=False)
    latencies.append(time.perf_counter()-start)
pd.DataFrame(dict(image=[p.name for p in latency_paths],seconds=latencies)).to_csv(ART/'cpu_latency.csv',index=False)
latency=dict(images=len(latencies),warmup=3,torch_threads=2,median_seconds=float(np.median(latencies)),p95_seconds=float(np.percentile(latencies,95)),scope='detector prediction only; not complete application latency')
summary=dict(status='complete',epochs_completed=len(history),train_images=8999,validation_images=1977,model='YOLOv8s',run=16,initial_checkpoint_sha256=EXPECTED_SHA256,external_training_overlap='unknown; not excluded',clean_generalization_claim_allowed=False,comparison_to_run14='different architecture and pretraining; pipeline comparison only',imgsz=640,detection=detection_metrics,image_macro_f1=macro,cpu_latency=latency,US_evaluated=False,local_evaluated=False,sample_sha256=manifest_digest,baseline_comparison='team v2 image assignments preserved; verify baseline implementation/threshold protocol before claiming controlled E1',pending_experiments=['E1 verified baseline comparison','E2 nano/small and 640/960','E3 augmentation ablation','E4 frozen-model US/local evaluation','E5 retrieval comparison owned by team'])
(ART/'summary.json').write_text(json.dumps(summary,indent=2))
weak=image_metrics.loc[image_metrics['mode'].eq('fixed_0.5')].sort_values('f1').iloc[0]
interpretation=(f"### Development results — interpretation\n\n"
    f"China detection mAP@50 is **{detection_metrics['metrics/mAP50(B)']:.2%}** and mAP@50–95 is **{detection_metrics['metrics/mAP50-95(B)']:.2%}**. "
    f"Image-level Macro-F1 is **{macro['fixed_0.5']:.2%}** at 0.5, or **{macro['validation_tuned_optimistic']:.2%}** after validation threshold tuning. "
    f"These answer different questions: detection evaluates boxes; image F1 evaluates damage presence. "
    f"The weakest image-level class at 0.5 is **{weak['cls']}** (F1 {weak['f1']:.2%}, recall {weak['recall']:.2%}).\n\n"
    f"Median detector CPU latency is **{latency['median_seconds']:.3f} s/image** for 20 images after warm-up, using two CPU threads. This excludes the rest of the app.\n\n"
    "China selected the checkpoint and tuned image thresholds. External pretraining overlap is unknown, so these are diagnostic development results, not independent accuracy or proof of local-road generalization. "
    "Compare against the baseline only after verifying matching image assignments and evaluation rules. Review missed potholes and false alarms before selecting the operational box threshold. "
    "RDD2022 US results cannot be assumed independent of external pretraining. Freeze the final recipe before an independently collected local evaluation. Local test images must never be used for training or hyperparameter selection.")
display(Markdown(interpretation))
(ART/'RESULTS_INTERPRETATION.md').write_text(interpretation)
print('RUN16_EVALUATION_COMPLETE',json.dumps(summary),flush=True)


Ultralytics 8.4.177 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2028.4±502.0 MB/s, size: 79.0 KB)
val: Scanning /kaggle/working/run16_team_v2_data/val/labels.cache... 1977 images, 43 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1977/1977 592.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 124/124 4.3it/s 28.9s
                   all       1977       4650       0.75        0.7      0.794      0.447
                   D00       1419       2678      0.779      0.414      0.681      0.379
                   D10        738       1096      0.656      0.822      0.769      0.404
                   D20        486        641      0.815      0.763      0.869      0.515
                   D40        164        235      0.749        0.8      0.856      0.489
Speed: 1.0ms preproces

,cls,AP50,AP50_95
0,D00,0.681415,0.378613
1,D10,0.769361,0.403842
2,D20,0.868668,0.515151
3,D40,0.855596,0.489248


,cls,mode,threshold,precision,recall,f1,support,always_positive_f1,auroc
0,D00,fixed_0.5,0.50,1.000000,0.196617,0.328622,1419,0.835689,0.967600
1,D10,fixed_0.5,0.50,0.949495,0.254743,0.401709,738,0.543646,0.977777
2,D20,fixed_0.5,0.50,0.996226,0.543210,0.703063,486,0.394641,0.985978
3,D40,fixed_0.5,0.50,0.969466,0.774390,0.861017,164,0.153199,0.988683
4,D00,validation_tuned_optimistic,0.05,0.944099,0.964059,0.953975,1419,0.835689,0.967600
5,D10,validation_tuned_optimistic,0.20,0.897338,0.959350,0.927308,738,0.543646,0.977777
6,D20,validation_tuned_optimistic,0.15,0.916488,0.880658,0.898216,486,0.394641,0.985978
7,D40,validation_tuned_optimistic,0.40,0.926174,0.841463,0.881789,164,0.153199,0.988683


### Development results — interpretation

China detection mAP@50 is **79.38%** and mAP@50–95 is **44.67%**. Image-level Macro-F1 is **57.36%** at 0.5, or **91.53%** after validation threshold tuning. These answer different questions: detection evaluates boxes; image F1 evaluates damage presence. The weakest image-level class at 0.5 is **D00** (F1 32.86%, recall 19.66%).

Median detector CPU latency is **0.222 s/image** for 20 images after warm-up, using two CPU threads. This excludes the rest of the app.

China selected the checkpoint and tuned image thresholds. External pretraining overlap is unknown, so these are diagnostic development results, not independent accuracy or proof of local-road generalization. Compare against the baseline only after verifying matching image assignments and evaluation rules. Review missed potholes and false alarms before selecting the operational box threshold. RDD2022 US results cannot be assumed independent of external pretraining. Freeze the final recipe before an independently collected local evaluation. Local test images must never be used for training or hyperparameter selection.

RUN16_EVALUATION_COMPLETE {"status": "complete", "epochs_completed": 17, "train_images": 8999, "validation_images": 1977, "model": "YOLOv8s", "run": 16, "initial_checkpoint_sha256": "565aae6915e34e4f7f4c44dac0bdcf9fff21b83c4adc74cc97ca09cc31882ce8", "external_training_overlap": "unknown; not excluded", "clean_generalization_claim_allowed": false, "comparison_to_run14": "different architecture and pretraining; pipeline comparison only", "imgsz": 640, "detection": {"metrics/precision(B)": 0.749665355972831, "metrics/recall(B)": 0.6998771963393245, "metrics/mAP50(B)": 0.7937601819357739, "metrics/mAP50-95(B)": 0.4467134595829971, "fitness": 0.4467134595829971}, "image_macro_f1": {"fixed_0.5": 0.5736027105528807, "validation_tuned_optimistic": 0.9153221600527851}, "cpu_latency": {"images": 20, "warmup": 3, "torch_threads": 2, "median_seconds": 0.22178126800008613, "p95_seconds": 0.22497208415020395, "scope": "detector prediction only; not complete application latency"}, "US_evaluated": fal

## 5. Prepare GitHub release assets and the experiment record

Package both run-14 and run-16 best checkpoints with distinct names, JSON results, hashes, attribution and a new experiment-table row. Preserve the actual metrics and limitations. Save a ZIP on Kaggle; do not publish incomplete training results. E2 and E3 remain separate follow-up experiments and are not run automatically.


In [5]:
import zipfile
RELEASE=Path('/kaggle/working/release_v2_yolo')
RELEASE.mkdir(exist_ok=True)
shutil.copy2(REF/'best.pt',RELEASE/'run14_yolo11s_best.pt')
shutil.copy2(ART/'best.pt',RELEASE/'run16_yolov8s_rdd_best.pt')
shutil.copy2(REF/'summary.json',RELEASE/'run14_results.json')
shutil.copy2(ART/'summary.json',RELEASE/'run16_results.json')
for name in ['provenance.json','class_mapping.json','settings_comparison_run14.json','inference_config.json','requested_training_settings.json','RESULTS_INTERPRETATION.md']:
    shutil.copy2(ART/name,RELEASE/name)
evaluation_dir=Path(det.save_dir)
plots=[p for p in evaluation_dir.glob('*.png')]
for p in plots:
    shutil.copy2(p,ART/p.name)
required_plots=dict(confusion=any('confusion_matrix' in p.name for p in plots),f1_confidence=any('F1_curve' in p.name for p in plots))
(ART/'plot_inventory.json').write_text(json.dumps(dict(directory=str(evaluation_dir),files=[p.name for p in plots],required_plots=required_plots),indent=2))
comparison=[]
for run,s in [(14,summary14),(16,summary)]:
    comparison.append(dict(run=run,model=s['model'],epochs=s['epochs_completed'],mAP50=s['detection']['metrics/mAP50(B)'],mAP50_95=s['detection']['metrics/mAP50-95(B)'],image_macro_f1_fixed_0_5=s['image_macro_f1']['fixed_0.5'],image_macro_f1_tuned_optimistic=s['image_macro_f1']['validation_tuned_optimistic']))
pd.DataFrame(comparison).to_csv(RELEASE/'run14_vs_run16.csv',index=False)
display(pd.DataFrame(comparison))
delta=summary['detection']['metrics/mAP50-95(B)']-summary14['detection']['metrics/mAP50-95(B)']
release_notes=(
    '# v2-yolo — RoadScan runs 14 and 16\n\n'
    'Run 14: YOLO11s initialized from COCO. Run 16: YOLOv8s initialized from SreekarAditya RDD2022 seed0 weights. '
    'Both use the same 8,999 team training images, 1,977 China MotorBike validation images and explicit training hyperparameters. '
    'Architecture and pretraining both differ; this is a pipeline comparison, not an isolated architecture or pretraining experiment. '
    'China validation selects checkpoints and image thresholds. External RDD2022 pretraining overlap is unknown. '
    'Scores do not establish Saudi-road generalization. Personal photos were not used in this run.\n\n'
    '## Assets\n'
    '- run14_yolo11s_best.pt: best checkpoint from run 14.\n'
    '- run16_yolov8s_rdd_best.pt: best checkpoint after run 16 fine-tuning.\n'
    '- run14_results.json and run16_results.json: actual evaluation results.\n'
    '- SHA256.json: release file hashes.\n\n'
    '## Attribution\n'
    'Published initialization: SreekarAditya, https://huggingface.co/SreekarAditya/yolo-rdd2022-benchmark (publisher CC BY 4.0). '
    'Data: RDD2022, Arya et al., CC BY 4.0. Team v2 sample: Sara; baseline reference: Fajer. '
    'Ultralytics software/model terms also apply; this archive does not relicense third-party material.\n'
)
(RELEASE/'RELEASE_NOTES.md').write_text(release_notes)
entry=(f"| 16 | YOLOv8s / published RDD2022 seed0, fine-tuned | 8,999 team v2 / China 1,977 | 640; same explicit hyperparameters as run 14 | "
       f"mAP50 {summary['detection']['metrics/mAP50(B)']:.6f}; mAP50-95 {summary['detection']['metrics/mAP50-95(B)']:.6f}; image Macro-F1@0.5 {summary['image_macro_f1']['fixed_0.5']:.6f} | "
       "Diagnostic development only; external overlap unknown; architecture differs from run 14 | run16_results.json; run16_yolov8s_rdd_best.pt (v2-yolo) |")
(RELEASE/'KAGGLE_EXPERIMENTS_RUN16_ROW.md').write_text(entry+'\n')
hashes={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in RELEASE.iterdir() if p.is_file() and p.name!='SHA256.json'}
(RELEASE/'SHA256.json').write_text(json.dumps(hashes,indent=2))
archive=shutil.make_archive('/kaggle/working/v2_yolo_release_assets','zip',root_dir=RELEASE)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    assert {'run14_yolo11s_best.pt','run16_yolov8s_rdd_best.pt','run14_results.json','run16_results.json'}<=set(z.namelist())
display(Markdown(f'### Run 14 vs run 16 — interpretation\n\nThe change in China mAP@50–95 is **{delta:+.2%}** (absolute difference). This comparison changes both architecture and pretraining. Unknown external pretraining overlap prevents a clean generalization claim. The JSON results, both best checkpoints and a GitHub experiment-table row are ready for the `v2-yolo` release. Publishing is a separate step after verifying the successful saved Kaggle output.'))
print('ALL_COMPLETE',json.dumps(dict(run=16,status='complete',release_archive=archive,release_files=sorted(p.name for p in RELEASE.iterdir()),plots=required_plots)),flush=True)


,run,model,epochs,mAP50,mAP50_95,image_macro_f1_fixed_0_5,image_macro_f1_tuned_optimistic
0,14,YOLO11s,64,0.227095,0.087325,0.244475,0.722955
1,16,YOLOv8s,17,0.793760,0.446713,0.573603,0.915322


### Run 14 vs run 16 — interpretation

The change in China mAP@50–95 is **+35.94%** (absolute difference). This comparison changes both architecture and pretraining. Unknown external pretraining overlap prevents a clean generalization claim. The JSON results, both best checkpoints and a GitHub experiment-table row are ready for the `v2-yolo` release. Publishing is a separate step after verifying the successful saved Kaggle output.

ALL_COMPLETE {"run": 16, "status": "complete", "release_archive": "/kaggle/working/v2_yolo_release_assets.zip", "release_files": ["KAGGLE_EXPERIMENTS_RUN16_ROW.md", "RELEASE_NOTES.md", "RESULTS_INTERPRETATION.md", "SHA256.json", "class_mapping.json", "inference_config.json", "provenance.json", "requested_training_settings.json", "run14_results.json", "run14_vs_run16.csv", "run14_yolo11s_best.pt", "run16_results.json", "run16_yolov8s_rdd_best.pt", "settings_comparison_run14.json"], "plots": {"confusion": true, "f1_confidence": true}}
